# 🐼 Pandas Basics — Exploring the Apple Store Dataset

**Pandas** is Python's most popular data library. It lets you load, explore, clean, and analyse data using a structure called a **DataFrame** — basically a table with rows and columns, just like a spreadsheet or a SQL table.

By the end of this notebook you'll know how to:
- Load CSV files into a DataFrame
- Explore and understand your data
- Select rows and columns
- Filter, sort, and rename
- Create new calculated columns
- Summarise data with `groupby`
- Merge / join multiple DataFrames
- Spot and handle missing values
- Make quick plots

We'll use the same Apple Store dataset from the SQL notebook — `products.csv`, `customers.csv`, and `sales.csv`. 🍎

## Lesson 1 — Loading Data

`pd.read_csv()` reads a CSV file and turns it into a **DataFrame**.  
A DataFrame is like a smart spreadsheet — rows are records, columns are fields.

In [ ]:
import pandas as pd

# Load all three tables
products  = pd.read_csv("products.csv")
customers = pd.read_csv("customers.csv")
sales     = pd.read_csv("sales.csv", parse_dates=["sale_date"])  # parse_dates makes the date column a proper date type

print("✅ Data loaded!")

## Lesson 2 — First Look at Your Data

These are the first things you do with ANY new dataset:

In [ ]:
# .head() shows the first 5 rows (the "head" of the table)
products.head()

In [ ]:
# .tail() shows the LAST 5 rows — useful to check the data doesn't cut off weirdly
sales.tail()

In [ ]:
# .shape tells you (rows, columns) — like asking "how big is this table?"
print(f"products  → {products.shape[0]:,} rows, {products.shape[1]} columns")
print(f"customers → {customers.shape[0]:,} rows, {customers.shape[1]} columns")
print(f"sales     → {sales.shape[0]:,} rows, {sales.shape[1]} columns")

In [ ]:
# .info() is the most informative first-look command
# It shows: column names, data types, and how many non-null (non-empty) values there are
products.info()

In [ ]:
# .describe() gives you stats for every numeric column at once:
# count, mean, std (standard deviation), min, 25th/50th/75th percentile, max
sales.describe()

In [ ]:
# .columns lists every column name — handy when you forget exact names
print(products.columns.tolist())

## Lesson 3 — Selecting Columns

There are two ways to pick a column:
- `df["column_name"]` → returns a **Series** (a single column)
- `df[["col1", "col2"]]` → returns a **DataFrame** (multiple columns)

In [ ]:
# Single column → Series (notice: single square brackets)
products["model"].head(10)

In [ ]:
# Multiple columns → DataFrame (notice: double square brackets [[ ]])
products[["model", "category", "price_usd"]].head(10)

## Lesson 4 — Selecting Rows with `.loc` and `.iloc`

- **`.iloc`** — select by **position** (like list indexing: 0, 1, 2 ...)
- **`.loc`**  — select by **label / condition**

In [ ]:
# .iloc — row 0 (the very first row)
products.iloc[0]

In [ ]:
# .iloc — rows 5 to 9 (slicing works just like Python lists!)
products.iloc[5:10]

In [ ]:
# .iloc — pick specific rows AND specific columns at the same time
# Format: .iloc[rows, columns]
products.iloc[0:5, 0:4]   # first 5 rows, first 4 columns

## Lesson 5 — Filtering Rows (Boolean Indexing)

This is one of the most used pandas skills. You write a **condition**, and pandas returns only the rows where it's `True`.

In [ ]:
# Filter: only iPhones
iphones = products[products["category"] == "iPhone"]
print(f"iPhone SKUs: {len(iphones)}")
iphones.head()

In [ ]:
# Filter: products over $2,000
expensive = products[products["price_usd"] > 2000]
expensive[["model", "category", "price_usd"]].sort_values("price_usd", ascending=False).head(10)

In [ ]:
# Multiple conditions — use & (AND) and | (OR)
# IMPORTANT: wrap each condition in parentheses ( )

# iPhones that are Blue AND under $1,200
blue_budget_iphones = products[
    (products["category"] == "iPhone") &
    (products["color"] == "Blue") &
    (products["price_usd"] < 1200)
]
blue_budget_iphones[["model", "storage", "color", "price_usd"]]

In [ ]:
# .isin() — match any value from a list (like SQL's IN)
mac_products = products[products["category"].isin(["Mac Laptop", "Mac Desktop"])]
print(f"Mac product variants: {len(mac_products)}")
mac_products[["model", "category", "variant_detail", "price_usd"]].head(10)

In [ ]:
# .str.contains() — like SQL's LIKE '%Pro%'
pro_models = products[products["model"].str.contains("Pro", case=False)]
pro_models[["model", "category", "price_usd"]].drop_duplicates("model")

## Lesson 6 — Sorting

`.sort_values()` sorts a DataFrame by one or more columns.

In [ ]:
# Sort by price — most expensive first
products[["model", "category", "price_usd"]].sort_values("price_usd", ascending=False).head(10)

In [ ]:
# Sort by multiple columns — category A→Z, then price highest→lowest within each category
products[["model", "category", "price_usd"]].sort_values(
    ["category", "price_usd"],
    ascending=[True, False]
).head(15)

## Lesson 7 — Creating New Columns

You can add calculated columns to a DataFrame just by assigning to a new column name.

In [ ]:
# Add a price_gbp column (using a made-up exchange rate of 0.79)
products["price_gbp"] = (products["price_usd"] * 0.79).round(2)

products[["model", "price_usd", "price_gbp"]].head(8)

In [ ]:
# Add a price_tier column using pd.cut() — bins values into labelled ranges
products["price_tier"] = pd.cut(
    products["price_usd"],
    bins=[0, 499, 1199, 2999, float("inf")],
    labels=["Budget", "Mid-Range", "Premium", "Ultra Premium"]
)

products[["model", "category", "price_usd", "price_tier"]].head(10)

In [ ]:
# Work with dates — extract year, month, day of week from sale_date
sales["year"]        = sales["sale_date"].dt.year
sales["month"]       = sales["sale_date"].dt.month
sales["day_of_week"] = sales["sale_date"].dt.day_name()

sales[["sale_date", "year", "month", "day_of_week", "total_price"]].head(8)

## Lesson 8 — Summarising with `value_counts()` and `groupby()`

### `value_counts()` — count how often each unique value appears

In [ ]:
# How many SKUs does each category have?
products["category"].value_counts()

In [ ]:
# Which sales channel do customers prefer?
sales["channel"].value_counts(normalize=True).mul(100).round(1).astype(str) + "%"

### `groupby()` — the pandas equivalent of SQL's GROUP BY

The pattern is always:
```python
df.groupby("column")["column_to_aggregate"].aggregation_function()
```

In [ ]:
# Total revenue per store — sorted highest first
sales.groupby("store")["total_price"].sum().sort_values(ascending=False)

In [ ]:
# Multiple aggregations at once with .agg()
sales.groupby("store")["total_price"].agg(
    total_revenue="sum",
    avg_order="mean",
    num_transactions="count"
).round(2).sort_values("total_revenue", ascending=False)

In [ ]:
# Group by multiple columns — revenue by year AND channel
sales.groupby(["year", "channel"])["total_price"].sum().unstack()

## Lesson 9 — Merging DataFrames (like SQL JOIN)

`pd.merge()` combines two DataFrames on a shared column.

```python
pd.merge(left_df, right_df, on="shared_column", how="inner")
```

`how=` options:
- `"inner"` — only rows that match in both tables *(default, like SQL JOIN)*
- `"left"`  — all rows from left, matched rows from right
- `"right"` — all rows from right, matched rows from left
- `"outer"` — all rows from both tables

In [ ]:
# Join sales with products to see what was sold
sales_with_product = pd.merge(sales, products, on="product_id", how="inner")

print(f"Rows after merge: {len(sales_with_product)}")
sales_with_product[["sale_date", "model", "category", "color", "quantity", "total_price"]].head(10)

In [ ]:
# Chain a groupby straight onto the merged DataFrame
# — Top 10 models by total revenue
sales_with_product.groupby(["category", "model"])["total_price"].sum()\
    .reset_index()\
    .sort_values("total_price", ascending=False)\
    .rename(columns={"total_price": "total_revenue"})\
    .head(10)

In [ ]:
# Now add customers too — three-table merge
full = pd.merge(sales_with_product, customers, on="customer_id", how="inner")

# Who are our top spending customers?
full.groupby(["customer_id", "first_name", "last_name", "city"])["total_price"].sum()\
    .reset_index()\
    .rename(columns={"total_price": "lifetime_spend"})\
    .sort_values("lifetime_spend", ascending=False)\
    .head(10)

## Lesson 10 — Renaming & Dropping Columns

In [ ]:
# Rename columns for readability
customers_clean = customers.rename(columns={
    "first_name": "First Name",
    "last_name":  "Last Name",
    "member_since": "Member Since",
})
customers_clean.head(3)

In [ ]:
# Drop columns you don't need
# axis=1 means "drop a column" (axis=0 would drop a row)
products_simple = products.drop(columns=["price_gbp", "price_tier"])
products_simple.head(3)

## Lesson 11 — Missing Values

Real data is messy — it often has gaps. Pandas represents missing values as `NaN` (Not a Number).

In [ ]:
# Let's create a small example with intentional missing values to demonstrate
import numpy as np

demo = pd.DataFrame({
    "product": ["iPhone 17", "AirPods Pro", "MacBook Air", "iPad Mini", "Apple Watch"],
    "units_sold": [120, 85, None, 60, None],
    "revenue":    [119880, 21165, None, 29940, None],
    "store":      ["Online", "New York", None, "Online", "Chicago"],
})
demo

In [ ]:
# .isna() — True where the value IS missing
demo.isna()

In [ ]:
# Count missing values per column
demo.isna().sum()

In [ ]:
# Option 1: fillna() — fill gaps with a default value
demo_filled = demo.fillna({"units_sold": 0, "revenue": 0, "store": "Unknown"})
demo_filled

In [ ]:
# Option 2: dropna() — remove rows that have ANY missing value
demo_dropped = demo.dropna()
demo_dropped

## Lesson 12 — Quick Plots with pandas

pandas has `.plot()` built in — it's great for quick exploratory charts.  
*(We use `matplotlib` under the hood — no need to import it separately for basic plots.)*

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-darkgrid")   # nicer looking charts

# Bar chart — total revenue by product category
category_revenue = sales_with_product.groupby("category")["total_price"].sum().sort_values(ascending=False)

category_revenue.plot(
    kind="bar",
    figsize=(10, 5),
    title="💰 Total Revenue by Product Category",
    ylabel="Revenue (USD)",
    color="steelblue",
    edgecolor="white",
)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Line chart — monthly revenue trend
monthly = sales.groupby(sales["sale_date"].dt.to_period("M"))["total_price"].sum()
monthly.index = monthly.index.astype(str)   # convert period to string for plotting

monthly.plot(
    kind="line",
    figsize=(12, 5),
    title="📈 Monthly Revenue Trend (2024–2025)",
    ylabel="Revenue (USD)",
    xlabel="Month",
    color="coral",
    linewidth=2,
    marker="o",
    markersize=4,
)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Horizontal bar chart — top 10 cities by number of orders
top_cities = full.groupby("city")["sale_id"].count().sort_values(ascending=True).tail(10)

top_cities.plot(
    kind="barh",
    figsize=(9, 5),
    title="🏙️ Top 10 Cities by Number of Orders",
    xlabel="Number of Orders",
    color="mediumseagreen",
    edgecolor="white",
)
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart — share of sales by channel
sales["channel"].value_counts().plot(
    kind="pie",
    figsize=(6, 6),
    title="🥧 Sales by Channel",
    autopct="%1.1f%%",
    colors=["steelblue", "coral"],
    startangle=90,
)
plt.ylabel("")   # hide the default y-label
plt.tight_layout()
plt.show()

---

## 🏋️ Challenges — Try These Yourself!

### Challenge 1
> **Filter the `sales` DataFrame to only include sales from 2025, then find the top 5 stores by total revenue for that year.**

*Hint: filter using `sales["year"] == 2025`, then `groupby` + `sum` + `sort_values`*

In [ ]:
# ✍️ Your code here:


<details>
<summary>👀 Click to reveal the answer</summary>

```python
sales_2025 = sales[sales["year"] == 2025]
sales_2025.groupby("store")["total_price"].sum().sort_values(ascending=False).head(5)
```
</details>

---

### Challenge 2
> **Using the `full` merged DataFrame, find the average age of customers who bought each product category.**

*Hint: `groupby("category")["age"].mean()` — then round to 1 decimal place*

In [ ]:
# ✍️ Your code here:


<details>
<summary>👀 Click to reveal the answer</summary>

```python
full.groupby("category")["age"].mean().round(1).sort_values()
```
</details>

---

### Challenge 3 — Bring it all together 🌟
> **Create a summary DataFrame called `monthly_summary` that shows, for each month (use `sale_date` period), the total revenue, number of transactions, and the best-selling product category. Then plot it as a bar chart.**

*Hint: This one needs a few steps — monthly revenue with groupby, then a separate groupby to find the top category per month using `.idxmax()` or `.mode()`*

In [ ]:
# ✍️ Your code here:


<details>
<summary>👀 Click to reveal the answer</summary>

```python
# Step 1: monthly revenue and transaction count
monthly_rev = sales_with_product.groupby(
    sales_with_product["sale_date"].dt.to_period("M")
).agg(
    total_revenue=("total_price", "sum"),
    transactions=("sale_id", "count"),
)

# Step 2: best-selling category per month
best_cat = sales_with_product.groupby(
    [sales_with_product["sale_date"].dt.to_period("M"), "category"]
)["total_price"].sum().reset_index()

best_cat = best_cat.loc[
    best_cat.groupby("sale_date")["total_price"].idxmax()
].set_index("sale_date")[["category"]].rename(columns={"category": "top_category"})

# Step 3: combine
monthly_summary = monthly_rev.join(best_cat)
print(monthly_summary.to_string())

# Step 4: plot
monthly_summary["total_revenue"].plot(
    kind="bar", figsize=(12, 5),
    title="Monthly Revenue with Transaction Count",
    color="steelblue", edgecolor="white"
)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
```
</details>

---

## 🎉 Pandas Cheat Sheet

| Task | Code |
|------|------|
| Load CSV | `pd.read_csv("file.csv")` |
| First/last rows | `df.head()` / `df.tail()` |
| Shape | `df.shape` |
| Column types | `df.info()` |
| Stats summary | `df.describe()` |
| Select column | `df["col"]` |
| Select columns | `df[["col1", "col2"]]` |
| Filter rows | `df[df["col"] == value]` |
| Multiple filters | `df[(cond1) & (cond2)]` |
| Contains text | `df["col"].str.contains("x")` |
| In a list | `df["col"].isin([...])` |
| Sort | `df.sort_values("col", ascending=False)` |
| New column | `df["new"] = df["a"] + df["b"]` |
| Rename | `df.rename(columns={"old": "new"})` |
| Drop column | `df.drop(columns=["col"])` |
| Count per group | `df["col"].value_counts()` |
| Group & sum | `df.groupby("col")["val"].sum()` |
| Multi-agg | `.agg(name=("col", "func"))` |
| Join tables | `pd.merge(df1, df2, on="key")` |
| Missing count | `df.isna().sum()` |
| Fill missing | `df.fillna(value)` |
| Drop missing | `df.dropna()` |
| Quick plot | `df.plot(kind="bar")` |

**You now know enough pandas to explore almost any real-world dataset. 🚀**